# UMUD v2 – Measure, Don't Regress

Pipeline: Segment → Geometry in original pixels → Real per-image scale → Proxy MAE validation

In [ ]:
# Cell 1: Environment & Paths
import os, math, cv2, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from scipy import ndimage
warnings.filterwarnings("ignore")

def seed_everything(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed); torch.backends.cudnn.deterministic = True

seed_everything(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] PyTorch {torch.__version__}  Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"[INFO] GPU: {torch.cuda.get_device_name(0)}")

def walk_find(root, target_name, is_dir=True):
    for cur, dirs, files in os.walk(root):
        haystack = dirs if is_dir else files
        if target_name in haystack:
            return os.path.join(cur, target_name)
    return None

ROOTS = ["/kaggle/input", ".", "/content"]
def find_path(name, is_dir=True):
    for r in ROOTS:
        if os.path.exists(r):
            p = walk_find(r, name, is_dir)
            if p: return p
    return None

APO_IMG_DIR  = find_path("apo_images_new_model_v1")
APO_MSK_DIR  = find_path("apo_masks_new_model_v1")
FASC_IMG_DIR = find_path("fasc_images_new_model_v1")
FASC_MSK_DIR = find_path("fasc_masks_new_model_v1")
TEST_IMG_DIR = find_path("test_set_v2")

# Search for scale CSV file
SCALE_CSV = None
for r in ROOTS:
    if not os.path.exists(r): continue
    for cur, dirs, files in os.walk(r):
        for f in files:
            if "universal_scale_metadata" in f.lower() and f.endswith(".csv"):
                SCALE_CSV = os.path.join(cur, f)
                break
        if SCALE_CSV: break

for k, v in [("APO_IMG", APO_IMG_DIR), ("APO_MSK", APO_MSK_DIR),
             ("FASC_IMG", FASC_IMG_DIR), ("FASC_MSK", FASC_MSK_DIR),
             ("TEST_IMG", TEST_IMG_DIR), ("SCALE_CSV", SCALE_CSV)]:
    print(f"[{k}] {v}")


In [ ]:
# Cell 2: Per-Image Scale Lookup
# Read the OCR-extracted scale CSV; build a stem -> mm_per_pixel dict.
# For test images (not in train) fall back to per-resolution estimate derived
# from the OCR findings.

def res_fallback_scale(w, h):
    """Fallback scale when image not in CSV."""
    if w == 1200 and h == 800:   return 50.0 / 720.0   # De 50mm, ~720 ROI px
    if w == 1640 and h == 1080:  return 45.0 / 980.0
    if w in (1088, 1069, 959):   return 45.0 / 600.0
    if 450 <= w <= 470:          return 45.0 / 460.0
    return 50.0 / max(h * 0.90, 1)

SCALE_LOOKUP = {}   # stem (e.g. "image_0023") -> mm_per_pixel
N_FROM_CSV   = 0

if SCALE_CSV and os.path.exists(SCALE_CSV):
    sdf = pd.read_csv(SCALE_CSV)
    print(f"Loaded scale CSV: {len(sdf)} rows, columns: {sdf.columns.tolist()}")

    for _, row in sdf.iterrows():
        stem = str(row["record_id"]).strip()
        # Use fasc depth first (more rows), then apo depth
        depth = None
        fasc_d = row.get("fasc_depth_mm")
        apo_d  = row.get("apo_depth_mm")
        if pd.notna(fasc_d) and float(fasc_d) > 0: depth = float(fasc_d)
        elif pd.notna(apo_d) and float(apo_d) > 0: depth = float(apo_d)

        # Use tick spacing if available (more direct)
        fasc_sp = row.get("fasc_spacing")
        apo_sp  = row.get("apo_spacing")
        if pd.notna(fasc_sp) and float(fasc_sp) > 0:
            # spacing is in pixels per 10mm tick
            SCALE_LOOKUP[stem] = 10.0 / float(fasc_sp)
            N_FROM_CSV += 1
        elif pd.notna(apo_sp) and float(apo_sp) > 0:
            SCALE_LOOKUP[stem] = 10.0 / float(apo_sp)
            N_FROM_CSV += 1
        elif depth is not None:
            # We don't know ROI height without image; store depth and resolve later
            SCALE_LOOKUP[stem] = ("depth", depth)

    print(f"  -> {N_FROM_CSV} entries with direct tick-based scale")
    print(f"  -> {len(SCALE_LOOKUP) - N_FROM_CSV} entries with depth only")
else:
    print("[WARN] Scale CSV not found – using resolution fallbacks for all images")

def get_scale(stem, orig_w, orig_h):
    """Return mm per pixel for this image."""
    val = SCALE_LOOKUP.get(stem)
    if val is None:
        return res_fallback_scale(orig_w, orig_h)
    if isinstance(val, tuple) and val[0] == "depth":
        depth = val[1]
        roi_h = orig_h * 0.90
        return depth / roi_h
    return float(val)


In [ ]:
# Cell 3: Dataset Index
def image_files(folder):
    if not folder or not os.path.exists(folder): return set()
    return {f for f in os.listdir(folder) if not f.lower().startswith("thumbs")}

apo_imgs  = image_files(APO_IMG_DIR)
apo_msks  = image_files(APO_MSK_DIR)
fasc_imgs = image_files(FASC_IMG_DIR)
fasc_msks = image_files(FASC_MSK_DIR)
test_imgs = image_files(TEST_IMG_DIR)

records = []
for fname in sorted(fasc_imgs):
    with Image.open(os.path.join(FASC_IMG_DIR, fname)) as im: w, h = im.size
    records.append(dict(image_id=fname, split="train_dual" if fname in apo_msks else "train_fasc_only",
                        has_apo=fname in apo_msks, has_fasc=True, orig_w=w, orig_h=h))

for fname in sorted(test_imgs):
    with Image.open(os.path.join(TEST_IMG_DIR, fname)) as im: w, h = im.size
    records.append(dict(image_id=fname, split="test", has_apo=False, has_fasc=False,
                        orig_w=w, orig_h=h))

meta = pd.DataFrame(records)
print(meta["split"].value_counts())
print("\nTest resolution breakdown:")
print(meta[meta["split"]=="test"].groupby(["orig_w","orig_h"])["image_id"].count())

# Count how many test images will use fallback scale
test_rows = meta[meta["split"]=="test"]
n_fallback = sum(1 for _, r in test_rows.iterrows()
                 if not isinstance(SCALE_LOOKUP.get(os.path.splitext(r["image_id"])[0]), (int, float)))
print(f"\nTest images using fallback scale: {n_fallback}/{len(test_rows)}")


In [ ]:
# Cell 4: PyTorch Dataset
# KEY FIX: 2:3 aspect ratio (384x576) so angles are not distorted.
# KEY FIX: cv2.INTER_AREA for masks to preserve thin 1-px fascicle lines.

IMG_H, IMG_W = 384, 576   # 2:3 – angles preserved

def load_img(path):
    img = np.array(Image.open(path).convert("L"), dtype=np.float32) / 255.0
    img = cv2.resize(img, (IMG_W, IMG_H), interpolation=cv2.INTER_LINEAR)
    return torch.tensor(img, dtype=torch.float32).unsqueeze(0)

def load_mask(path):
    m = np.array(Image.open(path).convert("L"), dtype=np.uint8)
    m = cv2.resize(m, (IMG_W, IMG_H), interpolation=cv2.INTER_AREA)
    return (m > 64).astype(np.float32)

def augment(img_np, apo_np, fasc_np):
    """img_np: HxW float32. Masks same shape. Returns augmented copies."""
    # Horizontal flip (safe: PA doesn't change)
    if np.random.rand() < 0.5:
        img_np  = img_np[:, ::-1].copy()
        apo_np  = apo_np[:, ::-1].copy()
        fasc_np = fasc_np[:, ::-1].copy()
    # Small rotation  ±8 deg
    angle = np.random.uniform(-8, 8)
    M = cv2.getRotationMatrix2D((IMG_W/2, IMG_H/2), angle, 1.0)
    img_np  = cv2.warpAffine(img_np,  M, (IMG_W, IMG_H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    apo_np  = cv2.warpAffine(apo_np,  M, (IMG_W, IMG_H), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT)
    fasc_np = cv2.warpAffine(fasc_np, M, (IMG_W, IMG_H), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT)
    # Brightness / contrast / gamma
    gamma = np.random.uniform(0.7, 1.3)
    img_np = np.clip(img_np ** gamma, 0, 1)
    img_np = np.clip(img_np * np.random.uniform(0.8, 1.2) + np.random.uniform(-0.1, 0.1), 0, 1)
    return img_np.astype(np.float32), apo_np.astype(np.float32), fasc_np.astype(np.float32)

class MuscleDataset(Dataset):
    def __init__(self, df, augment_on=False):
        self.df = df.reset_index(drop=True)
        self.augment_on = augment_on

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = row["image_id"]
        is_test = row["split"] == "test"

        img_dir = TEST_IMG_DIR if is_test else FASC_IMG_DIR
        img_t = load_img(os.path.join(img_dir, fname))

        if is_test:
            return img_t, fname, int(row["orig_w"]), int(row["orig_h"])

        apo_np  = load_mask(os.path.join(APO_MSK_DIR, fname)) if row["has_apo"] and APO_MSK_DIR else np.zeros((IMG_H, IMG_W), np.float32)
        fasc_np = load_mask(os.path.join(FASC_MSK_DIR, fname)) if row["has_fasc"] and FASC_MSK_DIR else np.zeros((IMG_H, IMG_W), np.float32)

        if self.augment_on:
            img_arr, apo_np, fasc_np = augment(img_t.squeeze().numpy(), apo_np, fasc_np)
            img_t = torch.tensor(img_arr).unsqueeze(0)

        masks = torch.tensor(np.stack([apo_np, fasc_np], 0), dtype=torch.float32)
        flags = torch.tensor([float(row["has_apo"]), float(row["has_fasc"])], dtype=torch.float32)
        return img_t, masks, flags


In [ ]:
# Cell 5: U-Net (ResNet34 Encoder) + Masked Loss
# Uses torchvision ResNet34 encoder pretrained on ImageNet.

import torchvision.models as tvm

class ConvBnRelu(nn.Module):
    def __init__(self, i, o, k=3, p=1):
        super().__init__()
        self.m = nn.Sequential(nn.Conv2d(i,o,k,padding=p,bias=False), nn.BatchNorm2d(o), nn.ReLU(True))
    def forward(self, x): return self.m(x)

class DecoderBlock(nn.Module):
    def __init__(self, in_c, skip_c, out_c):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_c, out_c, 2, stride=2)
        self.conv = nn.Sequential(ConvBnRelu(out_c+skip_c, out_c), ConvBnRelu(out_c, out_c))
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = nn.functional.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], 1))

class UNetR34(nn.Module):
    def __init__(self):
        super().__init__()
        enc = tvm.resnet34(weights=tvm.ResNet34_Weights.DEFAULT)
        # adapt first conv for grayscale (1 channel) by averaging the 3-channel weights
        w = enc.conv1.weight.data.mean(dim=1, keepdim=True)
        enc.conv1 = nn.Conv2d(1, 64, 7, stride=2, padding=3, bias=False)
        enc.conv1.weight.data = w

        self.e0 = nn.Sequential(enc.conv1, enc.bn1, enc.relu)   # 64, /2
        self.pool = enc.maxpool                                   # /4
        self.e1 = enc.layer1   # 64,  /4
        self.e2 = enc.layer2   # 128, /8
        self.e3 = enc.layer3   # 256, /16
        self.e4 = enc.layer4   # 512, /32

        self.d4 = DecoderBlock(512, 256, 256)
        self.d3 = DecoderBlock(256, 128, 128)
        self.d2 = DecoderBlock(128,  64,  64)
        self.d1 = DecoderBlock( 64,  64,  32)

        self.up_final = nn.ConvTranspose2d(32, 32, 2, stride=2)
        self.head_apo  = nn.Conv2d(32, 1, 1)
        self.head_fasc = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e0 = self.e0(x)
        ep = self.pool(e0)
        e1 = self.e1(ep)
        e2 = self.e2(e1)
        e3 = self.e3(e2)
        e4 = self.e4(e3)

        d = self.d4(e4, e3)
        d = self.d3(d,  e2)
        d = self.d2(d,  e1)
        d = self.d1(d,  e0)
        d = self.up_final(d)

        return torch.cat([self.head_apo(d), self.head_fasc(d)], dim=1)   # logits


class MaskedSegLoss(nn.Module):
    """BCE + Dice; extra positive weight on fascicle channel (sparse lines)."""
    def __init__(self, fasc_pos_weight=5.0, smooth=1e-5):
        super().__init__()
        self.fasc_pos_weight = fasc_pos_weight
        self.smooth = smooth

    def _bce_dice(self, logits, targets, pos_weight=1.0):
        pw = torch.tensor(pos_weight, device=logits.device)
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=pw, reduction="mean")
        prob = torch.sigmoid(logits)
        inter = (prob * targets).sum()
        union = prob.sum() + targets.sum()
        dice = 1.0 - (2*inter + self.smooth) / (union + self.smooth)
        return bce + dice

    def forward(self, logits, targets, flags):
        # logits, targets: (B, 2, H, W); flags: (B, 2)
        loss = torch.tensor(0.0, device=logits.device)
        count = 0
        for c, pw in enumerate([1.0, self.fasc_pos_weight]):
            mask = flags[:, c].bool()
            if not mask.any(): continue
            loss += self._bce_dice(logits[mask, c], targets[mask, c], pw)
            count += 1
        return loss / max(count, 1)


In [ ]:
# Cell 6: Training
train_df = meta[meta["split"] != "test"].sample(frac=1, random_state=42).reset_index(drop=True)
val_df   = train_df.iloc[:250].reset_index(drop=True)
train_df = train_df.iloc[250:].reset_index(drop=True)

train_ds = MuscleDataset(train_df, augment_on=True)
val_ds   = MuscleDataset(val_df,   augment_on=False)

train_dl = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

model     = UNetR34().to(DEVICE)
criterion = MaskedSegLoss(fasc_pos_weight=5.0)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)

EPOCHS = 40
best_val_dice = -1
CKPT = "/kaggle/working/best_model.pth"

def dice_score(pred_prob, target, thresh=0.5):
    pred = (pred_prob > thresh).float()
    inter = (pred * target).sum()
    return (2*inter + 1e-5) / (pred.sum() + target.sum() + 1e-5)

print(f"Training {EPOCHS} epochs on {len(train_ds)} images, validating on {len(val_ds)}...")

for epoch in range(1, EPOCHS+1):
    model.train()
    run_loss = 0.0
    for imgs, masks, flags in train_dl:
        imgs, masks, flags = imgs.to(DEVICE), masks.to(DEVICE), flags.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, masks, flags)
        loss.backward()
        optimizer.step()
        run_loss += loss.item() * len(imgs)
    scheduler.step()

    if epoch % 5 == 0 or epoch == 1:
        model.eval()
        apo_dice_list, fasc_dice_list = [], []
        with torch.no_grad():
            for imgs, masks, flags in val_dl:
                imgs, masks, flags = imgs.to(DEVICE), masks.to(DEVICE), flags.to(DEVICE)
                probs = torch.sigmoid(model(imgs))
                for c, lst in [(0, apo_dice_list), (1, fasc_dice_list)]:
                    m = flags[:, c].bool()
                    if m.any():
                        lst.append(dice_score(probs[m, c], masks[m, c]).item())

        apo_d  = np.mean(apo_dice_list)  if apo_dice_list  else 0
        fasc_d = np.mean(fasc_dice_list) if fasc_dice_list else 0
        mean_d = (apo_d + fasc_d) / 2
        lr = scheduler.get_last_lr()[0]
        print(f"Ep {epoch:03d} | Loss {run_loss/len(train_ds):.4f} | "
              f"Dice apo={apo_d:.3f} fasc={fasc_d:.3f} | LR={lr:.6f}")

        if mean_d > best_val_dice:
            best_val_dice = mean_d
            torch.save(model.state_dict(), CKPT)
            print(f"  ** Saved best checkpoint (mean dice={mean_d:.3f})")

print(f"\nTraining done. Best val dice = {best_val_dice:.4f}")
model.load_state_dict(torch.load(CKPT))
print("Best checkpoint loaded.")


In [ ]:
# Cell 7: Geometric Measurement Engine
# All geometry is done in ORIGINAL IMAGE pixel space (not the resized grid).

def fit_line_robust(ys, xs):
    """Fit y = m*x + b with RANSAC-like trimming. Returns (m, b) or None."""
    if len(xs) < 10: return None
    m, b = np.polyfit(xs, ys, 1)
    pred = m * xs + b
    res  = np.abs(ys - pred)
    keep = res < np.percentile(res, 75)
    if keep.sum() < 10: return None
    return np.polyfit(xs[keep], ys[keep], 1)   # (m, b)

def line_angle_deg(m):
    """Angle of y=mx+b in degrees, in range (-90, 90]."""
    return math.degrees(math.atan(m))

def signed_pa(theta_fasc, theta_apo):
    """PA = angle between fascicle and deep apo, wrapped to 0–90 degrees."""
    diff = theta_fasc - theta_apo
    # resolve 180-deg ambiguity
    while diff >  90: diff -= 180
    while diff < -90: diff += 180
    return abs(diff)

def line_intersection_y(m, b, y_target):
    """Find x where y = m*x+b equals y_target: x = (y_target - b) / m"""
    if abs(m) < 1e-9: return None
    return (y_target - b) / m

def measure_image(apo_prob, fasc_prob, orig_w, orig_h):
    """
    apo_prob, fasc_prob: numpy (IMG_H, IMG_W) float32 probability maps.
    Returns (pa_deg, fl_mm, mt_mm).
    """
    # Upsample predictions back to original image space
    apo_full  = cv2.resize(apo_prob,  (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)
    fasc_full = cv2.resize(fasc_prob, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)

    apo_bin  = (apo_full  > 0.4).astype(np.uint8)
    fasc_bin = (fasc_full > 0.4).astype(np.uint8)

    # ── Aponeuroses ──────────────────────────────────────────────────────────
    apo_labels, n_apo = ndimage.label(apo_bin)
    apo_lines = []
    if n_apo >= 2:
        sizes = [(apo_labels == i).sum() for i in range(1, n_apo+1)]
        top2  = sorted(range(1, n_apo+1), key=lambda i: sizes[i-1], reverse=True)[:2]
        # sort by centroid y: smaller y = superficial
        top2  = sorted(top2, key=lambda i: np.where(apo_labels==i)[0].mean())
        for lbl in top2:
            ys, xs = np.where(apo_labels == lbl)
            line = fit_line_robust(ys.astype(float), xs.astype(float))
            if line: apo_lines.append(line)
    elif n_apo == 1:
        ys, xs = np.where(apo_bin > 0)
        mid_y  = ys.mean()
        for sel in [ys < mid_y, ys >= mid_y]:
            if sel.sum() > 10:
                line = fit_line_robust(ys[sel].astype(float), xs[sel].astype(float))
                if line: apo_lines.append(line)

    if len(apo_lines) < 2:
        return None   # can't measure

    sup_m, sup_b = apo_lines[0]   # superficial (smaller y)
    dep_m, dep_b = apo_lines[1]   # deep  (larger  y)
    theta_apo_sup = line_angle_deg(sup_m)
    theta_apo_dep = line_angle_deg(dep_m)
    theta_apo     = (theta_apo_sup + theta_apo_dep) / 2.0

    # ── MT in pixels ─────────────────────────────────────────────────────────
    x_min = max(0, int(orig_w * 0.1))
    x_max = min(orig_w, int(orig_w * 0.9))
    xs    = np.linspace(x_min, x_max, 100)
    y_sup = sup_m * xs + sup_b
    y_dep = dep_m * xs + dep_b
    mt_px = np.mean(np.abs(y_dep - y_sup))

    # ── Fascicles ─────────────────────────────────────────────────────────────
    fasc_labels, n_fasc = ndimage.label(fasc_bin)
    pa_list, fl_list = [], []

    for lbl in range(1, n_fasc+1):
        ys, xs = np.where(fasc_labels == lbl)
        if len(xs) < 15: continue
        line = fit_line_robust(ys.astype(float), xs.astype(float))
        if line is None: continue
        fm, fb = line
        theta_fasc = line_angle_deg(fm)
        pa = signed_pa(theta_fasc, theta_apo_dep)

        # Skip fascicles nearly parallel to aponeurosis (intersection unstable)
        if pa < 3.0: continue

        # FL: intersect fascicle with sup and dep aponeurosis lines
        # Parametrize fascicle: y = fm*x + fb; sup: y = sup_m*x + sup_b -> solve
        denom_sup = fm - sup_m
        denom_dep = fm - dep_m
        if abs(denom_sup) < 1e-6 or abs(denom_dep) < 1e-6: continue

        x_int_sup = (sup_b - fb) / denom_sup
        x_int_dep = (dep_b - fb) / denom_dep
        y_int_sup = fm * x_int_sup + fb
        y_int_dep = fm * x_int_dep + fb

        fl_px = math.hypot(x_int_dep - x_int_sup, y_int_dep - y_int_sup)

        # Sanity: fl must be larger than mt and the intersection points plausible
        if fl_px < mt_px * 0.5 or fl_px > orig_w * 2: continue

        pa_list.append(pa)
        fl_list.append(fl_px)

    return pa_list, fl_list, mt_px, theta_apo

# Collect training-mask measurements to use as fallback medians
print("Computing fallback statistics from training ground-truth masks...")

fallback_pa_vals, fallback_fl_vals, fallback_mt_vals = [], [], []
SAMPLE = 300
sample_rows = meta[meta["split"].isin(["train_dual", "train_fasc_only"])].sample(
    min(SAMPLE, len(meta)), random_state=0)

for _, row in sample_rows.iterrows():
    fname = row["image_id"]
    orig_w, orig_h = int(row["orig_w"]), int(row["orig_h"])
    if not (row["has_apo"] and row["has_fasc"]): continue
    apo_p  = np.array(Image.open(os.path.join(APO_MSK_DIR,  fname)).convert("L"), dtype=np.float32) / 255.0
    fasc_p = np.array(Image.open(os.path.join(FASC_MSK_DIR, fname)).convert("L"), dtype=np.float32) / 255.0
    result = measure_image(apo_p, fasc_p, orig_w, orig_h)
    if result is None: continue
    pa_list, fl_list, mt_px, _ = result
    if pa_list:
        stem  = os.path.splitext(fname)[0]
        scale = get_scale(stem, orig_w, orig_h)
        fallback_pa_vals.extend(pa_list)
        fallback_fl_vals.extend([f * scale for f in fl_list])
        fallback_mt_vals.append(mt_px * scale)

FALLBACK_PA = float(np.median(fallback_pa_vals)) if fallback_pa_vals else 17.0
FALLBACK_FL = float(np.median(fallback_fl_vals)) if fallback_fl_vals else 70.0
FALLBACK_MT = float(np.median(fallback_mt_vals)) if fallback_mt_vals else 21.0
print(f"Fallback PA={FALLBACK_PA:.2f}  FL={FALLBACK_FL:.2f}  MT={FALLBACK_MT:.2f}")


In [ ]:
# Cell 8: Proxy MAE Validation on Ground-Truth Masks
# Run geometry on GT masks vs predicted masks to separate segmentation error
# from geometry/scale error.

print("Running proxy validation on val split ground-truth masks...")

model.eval()
proxy_rows = val_df[val_df["has_apo"] & val_df["has_fasc"]].reset_index(drop=True)

gt_pa, gt_fl, gt_mt = [], [], []
pred_pa, pred_fl, pred_mt = [], [], []

for _, row in proxy_rows.iterrows():
    fname  = row["image_id"]
    orig_w, orig_h = int(row["orig_w"]), int(row["orig_h"])
    stem   = os.path.splitext(fname)[0]
    scale  = get_scale(stem, orig_w, orig_h)

    # Ground-truth geometry
    apo_gt  = np.array(Image.open(os.path.join(APO_MSK_DIR,  fname)).convert("L"), dtype=np.float32)/255.
    fasc_gt = np.array(Image.open(os.path.join(FASC_MSK_DIR, fname)).convert("L"), dtype=np.float32)/255.
    res_gt  = measure_image(apo_gt, fasc_gt, orig_w, orig_h)

    # Predicted geometry
    img_t = load_img(os.path.join(FASC_IMG_DIR, fname)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.sigmoid(model(img_t)).cpu().numpy()[0]
    res_pr = measure_image(probs[0], probs[1], orig_w, orig_h)

    if res_gt is None or res_pr is None: continue
    pa_gt_l, fl_gt_l, mt_gt_px, _ = res_gt
    pa_pr_l, fl_pr_l, mt_pr_px, _ = res_pr

    if not pa_gt_l or not pa_pr_l: continue
    gt_pa.append(np.median(pa_gt_l))
    gt_fl.append(np.median(fl_gt_l) * scale)
    gt_mt.append(mt_gt_px * scale)
    pred_pa.append(np.median(pa_pr_l))
    pred_fl.append(np.median(fl_pr_l) * scale)
    pred_mt.append(mt_pr_px * scale)

if gt_pa:
    mae_pa = np.mean(np.abs(np.array(gt_pa) - np.array(pred_pa)))
    mae_fl = np.mean(np.abs(np.array(gt_fl) - np.array(pred_fl)))
    mae_mt = np.mean(np.abs(np.array(gt_mt) - np.array(pred_mt)))
    print(f"\nProxy MAE on {len(gt_pa)} images:")
    print(f"  PA: {mae_pa:.3f} deg")
    print(f"  FL: {mae_fl:.3f} mm")
    print(f"  MT: {mae_mt:.3f} mm")
else:
    print("Not enough paired GT+Pred results for proxy MAE.")


In [ ]:
# Cell 9: Test Inference & Submission
test_df_sub = meta[meta["split"] == "test"].reset_index(drop=True)
test_ds = MuscleDataset(test_df_sub, augment_on=False)
test_dl = DataLoader(test_ds, batch_size=4, shuffle=False, num_workers=2)

model.eval()
rows = []

print(f"Running inference on {len(test_df_sub)} test images...")

for imgs, fnames, orig_ws, orig_hs in test_dl:
    imgs = imgs.to(DEVICE)
    with torch.no_grad():
        probs = torch.sigmoid(model(imgs)).cpu().numpy()

    for i, fname in enumerate(fnames):
        orig_w = int(orig_ws[i]); orig_h = int(orig_hs[i])
        stem   = os.path.splitext(fname)[0]
        scale  = get_scale(stem, orig_w, orig_h)

        result = measure_image(probs[i, 0], probs[i, 1], orig_w, orig_h)

        if result is not None and result[0]:
            pa_list, fl_list, mt_px, _ = result
            pa = float(np.median(pa_list))
            fl = float(np.median(fl_list)) * scale
            mt = mt_px * scale
        else:
            pa, fl, mt = FALLBACK_PA, FALLBACK_FL, FALLBACK_MT

        rows.append({"image_id": fname, "pa_deg": round(pa, 3),
                     "fl_mm": round(fl, 3), "mt_mm": round(mt, 3)})

sub = pd.DataFrame(rows)
sub.to_csv("/kaggle/working/submission.csv", index=False, sep=",")
print(f"[SUCCESS] submission.csv written with {len(sub)} rows")
print(sub.head(10))
print("\nVerifying first 5 raw lines:")
with open("/kaggle/working/submission.csv") as f:
    for _ in range(5): print(f.readline().strip())
